In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm
import re
import sys
from pathlib import Path
import glob
import subprocess

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [2]:
# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing_local import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities_local import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


In [5]:
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'
local_path = "local-files"

###

EVENT_NAME = "202408_TropicalStorm_Debby"
product = "landsat9"

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

keys = [x.split("/")[-1] for x in get_all_s3_keys(s3_client, BUCKET, f"drcs_activations/{EVENT_NAME}/{product}", ".tif")] if s3_client else []

keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files


['LC09_colorInfrared_20240730_155953_017037.tif',
 'LC09_colorInfrared_20240806_160630_018038.tif',
 'LC09_colorInfrared_20240806_160654_018039.tif',
 'LC09_colorInfrared_20240806_160717_018040.tif',
 'LC09_colorInfrared_20240808_155432_016039.tif',
 'LC09_colorInfrared_20240808_155456_016040.tif',
 'LC09_colorInfrared_20240808_15548_016038.tif',
 'LC09_colorInfrared_20240808_155520_016041.tif',
 'LC09_colorInfrared_20240808_155544_016042.tif',
 'LC09_naturalColor_20240730_155953_017037.tif',
 'LC09_naturalColor_20240806_160630_018038.tif',
 'LC09_naturalColor_20240806_160654_018039.tif',
 'LC09_naturalColor_20240806_160717_018040.tif',
 'LC09_naturalColor_20240808_155432_016039.tif',
 'LC09_naturalColor_20240808_155456_016040.tif',
 'LC09_naturalColor_20240808_15548_016038.tif',
 'LC09_naturalColor_20240808_155520_016041.tif',
 'LC09_naturalColor_20240808_155544_016042.tif',
 'LC09_trueColor_20240730_155953_017037.tif',
 'LC09_trueColor_20240806_160630_018038.tif',
 'LC09_trueColor_20

In [5]:
# Download the desired files to a local directory with a known path
local_file_dir = os.path.abspath(f"./{local_path}")
if not os.path.exists(local_file_dir):
    os.mkdir(local_file_dir)
for key in keys:
    subprocess.run([
        "aws",
        "s3",
        "cp",
        f"s3://nasa-disasters/drcs_activations/{EVENT_NAME}/{product}/{key}",
        local_file_dir], check = True)

download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/landsat/LC08_colorInfrared_20240902_16388_023039.tif to local-files/LC08_colorInfrared_20240902_16388_023039.tif
download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/landsat/LC08_colorInfrared_20240909_164421_024039.tif to local-files/LC08_colorInfrared_20240909_164421_024039.tif
download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/landsat/LC08_trueColor_20240902_16388_023039.tif to local-files/LC08_trueColor_20240902_16388_023039.tif
download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/landsat/LC08_trueColor_20240909_164421_024039.tif to local-files/LC08_trueColor_20240909_164421_024039.tif
download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/landsat/LC09_colorInfrared_20240908_165024_025039.tif to local-files/LC09_colorInfrared_20240908_165024_025039.tif
download: s3://nasa-disasters/drcs_activations/202409_Tropic

In [6]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()


# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

In [7]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [8]:
def make_regex_dict(keys, regexes, products):
    ret = {}
    for i in range(len(products)):
        matches = []
        for key in keys:
            filename = key.split("/")[-1]
            match = re.search(regexes[i], filename)
            if match is not None:
                matches.append(key)
        if matches != []:
            ret[products[i]] = matches
    return ret

In [9]:
def create_cog_filename(filename, event):
    sname = filename.split("/")[-1].replace(".tif", "").split("_")
    date = datetime.strptime(sname[2], "%Y%m%d")
    new_dt_format = date.strftime("%Y-%m-%d_day")
    cog_filename = f"{event}_{sname[0]}_{sname[1]}_{sname[3]}_{new_dt_format}.tif"
    return cog_filename

In [10]:
def simple_process_files(file_list, rename_func, target_dir, event):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    print("Testing filenams:")
    for filename in file_list:
        print(f"  {rename_func(filename, event)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        #"raw_data_bucket": BUCKET,
        #"raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{event}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(local_download_path, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            local_download_path, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=file_list,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=event,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

In [15]:
local_keys = [x for x in glob.glob(f"{local_path}/*") if x.endswith(".tif")]

reg_keys = make_regex_dict(local_keys, [r".*_colorInfrared_.*.tif", r".*_trueColor_.*.tif", r".*_naturalColor_.*.tif"], ["colorInfrared", "trueColor", "naturalColor"])

In [16]:
print(reg_keys)
for k, v in reg_keys.items():
    for filename in v:
        print(create_cog_filename(filename, EVENT_NAME))

{'colorInfrared': ['local-files/LC08_colorInfrared_20240902_16388_023039.tif', 'local-files/LC08_colorInfrared_20240909_164421_024039.tif', 'local-files/LC09_colorInfrared_20240908_165024_025039.tif'], 'trueColor': ['local-files/LC08_trueColor_20240902_16388_023039.tif', 'local-files/LC08_trueColor_20240909_164421_024039.tif', 'local-files/LC09_trueColor_20240908_165024_025039.tif']}
202409_TropicalStorm_Francine_LC08_colorInfrared_16388_2024-09-02_day.tif
202409_TropicalStorm_Francine_LC08_colorInfrared_164421_2024-09-09_day.tif
202409_TropicalStorm_Francine_LC09_colorInfrared_165024_2024-09-08_day.tif
202409_TropicalStorm_Francine_LC08_trueColor_16388_2024-09-02_day.tif
202409_TropicalStorm_Francine_LC08_trueColor_164421_2024-09-09_day.tif
202409_TropicalStorm_Francine_LC09_trueColor_165024_2024-09-08_day.tif


In [17]:
for k, v in reg_keys.items():
    results = simple_process_files(file_list = v, rename_func = create_cog_filename, target_dir = f"Landsat/{k}", event = EVENT_NAME)

Testing filenams:
  202409_TropicalStorm_Francine_LC08_colorInfrared_16388_2024-09-02_day.tif
  202409_TropicalStorm_Francine_LC08_colorInfrared_164421_2024-09-09_day.tif
  202409_TropicalStorm_Francine_LC09_colorInfrared_165024_2024-09-08_day.tif
Configuration loaded:
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Landsat/colorInfrared

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202409_TropicalStorm_Francine

[1/3] Processing: local-files/LC08_colorInfrared_20240902_16388_023039.tif
   Output filename: 202409_TropicalStorm_Francine_LC08_colorInfrared_16388_2024-09-02_day.tif
   [CACHE HIT] Using local file: local-files/LC08_colorInfrared_20240902_16388_023039.tif
   [MEMORY] Initial: 295.9 MB
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 3.00 MB
   [NODATA] RGB file detected with nodata=0, treating as regular RGB without nodata
   [CHUNKS] Processing 72 c

   [BAND 2/3] Processing...


Band 2:  67%|██████▋   | 48/72 [00:01<00:00, 28.11chunks/s]


   [MEMORY] High usage: 591.8 MB, forcing cleanup...


Band 2:  81%|████████  | 58/72 [00:02<00:00, 27.57chunks/s]

...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=4, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=4, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=4, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmp3zmvnt_a_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp7owlubux.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/202409_TropicalStorm_Francine_LC09_trueColor_165024_2024-09-08_day.tif
   [MEMORY] Final: 840.8 MB (Change: -52.8 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_TropicalStorm_Francine_LC09_trueColor_165024_2024-09-08_day.tif

✅ Batch processing complete: 3 files processed
📁 COGs saved locally to: output/202409_TropicalStorm_Francine

📊 BATCH PROCESSING SUMMARY
Total files processed: 3
Successful: 3
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-25T17:28:36.869012


In [22]:
subprocess.run(["rm", "-r", f"{local_path}"], check = True)
subprocess.run(["rm", "-r", os.path.abspath("./output")], check = True)
subprocess.run(["rm", "-r", os.path.abspath("./reproj")], check = True)

CompletedProcess(args=['rm', '-r', '/home/jovyan/conversion_scripts/convert-files-and-move/2024/reproj'], returncode=0)